# 07 Genie Agent: create or update the Genie Space

Creates (or re-applies) the one Genie Space the PM Co-Pilot chat uses: **Blue Ridge LNG — PM Co-Pilot (synthetic)**. All data is synthetic.

**How the space is defined.** CLI v1.20.0's bundle schema has a `genie_spaces` resource, so the space is declared in `resources/genie-space.yml` and created by `databricks bundle deploy -t demo`. The definition (11 curated tables with descriptions and synonyms, one general instruction block, 10 example Question-to-SQL pairs) lives in `genie_space.geniespace.json`, generated by `build_space.py`, which also runs every verified SQL against the warehouse to write `sample_questions.md` and `heldout_questions.md`.

**What this notebook adds.** An idempotent create-or-update by title: it pages through the workspace's Genie spaces, finds the one with this exact title, and re-applies the definition file with `PATCH` (or creates it with `POST` if the bundle has not been deployed yet). It never touches another space, and it never creates a second one. It then checks the saved space, confirms none of the five held-out benchmark questions are in it, and writes the space id to `outputs/space_id.txt`.

**Tables in the space:** `bluerridge_gold_pm.{compliance_findings, kpi_daily, risk_forecast, schedule_overdue, drawing_register, compliance_eval, daily_brief}`, `bluerridge_silver_engineering.{schedule, ncrs, change_orders}` and `bluerridge_agents.code_excerpts`. Left out on purpose: raw and staging tables (`compliance_judge_raw*`, `drawings_extracted_raw`), `compliance_suppressed`, `ground_truth_findings`, the `risk_series*` tables and `discipline_access`.

**Identity.** This notebook runs as the submitting user. In the app, Genie is called on behalf of the signed-in PM (OBO, `dashboards.genie` scope), so every query runs with that user's Unity Catalog permissions, including the row filter on `drawing_register`.

In [1]:
import json, os
from databricks.sdk import WorkspaceClient

dbutils.widgets.text("run_id", "", "Job run id")
RUN_ID = dbutils.widgets.get("run_id")
print("job run id:", RUN_ID or "(interactive)")

w = WorkspaceClient()
print("running as:", w.current_user.me().user_name)

TITLE = "Blue Ridge LNG — PM Co-Pilot (synthetic)"
DESCRIPTION = ("Ask about drawing compliance, schedule, risk, cost, safety and NCRs for the Blue Ridge LNG Terminal Phase 2 build. "
               "All data is synthetic (demo).")
WAREHOUSE_ID = "527214d7319cb124"

job run id: 793283877903127
running as: joseph.hobbs@databricks.com


## 1. Load the space definition
The JSON file sits next to this notebook (uploaded with it).

In [2]:
# The workspace refuses to import a file ending in ".geniespace.json", so the scratch copy is named genie_space.json.
DEF_FILE = next(p for p in ("genie_space.geniespace.json", "genie_space.json") if os.path.exists(p))
print("definition file:", DEF_FILE)
space = json.load(open(DEF_FILE))
tables = [t["identifier"] for t in space["data_sources"]["tables"]]
examples = space["instructions"]["example_question_sqls"]
print(f"{len(tables)} tables, {len(examples)} example SQLs, {len(space['instructions']['text_instructions'])} instruction block")
for t in tables:
    print("  ", t)
serialized = json.dumps(space, ensure_ascii=False)

definition file: genie_space.json
11 tables, 10 example SQLs, 1 instruction block
   mfg_mc_se_sa.bluerridge_agents.code_excerpts
   mfg_mc_se_sa.bluerridge_gold_pm.compliance_eval
   mfg_mc_se_sa.bluerridge_gold_pm.compliance_findings
   mfg_mc_se_sa.bluerridge_gold_pm.daily_brief
   mfg_mc_se_sa.bluerridge_gold_pm.drawing_register
   mfg_mc_se_sa.bluerridge_gold_pm.kpi_daily
   mfg_mc_se_sa.bluerridge_gold_pm.risk_forecast
   mfg_mc_se_sa.bluerridge_gold_pm.schedule_overdue
   mfg_mc_se_sa.bluerridge_silver_engineering.change_orders
   mfg_mc_se_sa.bluerridge_silver_engineering.ncrs
   mfg_mc_se_sa.bluerridge_silver_engineering.schedule


## 2. Find the space by title, then create or update
Paginates through every Genie space in the workspace and matches the exact title only.

In [3]:
def find_spaces_by_title(title):
    found, token = [], None
    while True:
        params = {"page_size": 100}
        if token:
            params["page_token"] = token
        resp = w.api_client.do("GET", "/api/2.0/genie/spaces", query=params)
        found += [s for s in resp.get("spaces", []) if s.get("title") == title]
        token = resp.get("next_page_token")
        if not token:
            return found

matches = find_spaces_by_title(TITLE)
print("spaces with this exact title:", [m["space_id"] for m in matches])
assert len(matches) <= 1, "more than one space has this title; refusing to guess which to update"

body = {"title": TITLE, "description": DESCRIPTION, "warehouse_id": WAREHOUSE_ID, "serialized_space": serialized}
if matches:
    SPACE_ID = matches[0]["space_id"]
    w.api_client.do("PATCH", f"/api/2.0/genie/spaces/{SPACE_ID}", body=body)
    print("UPDATED existing space", SPACE_ID)
else:
    user = w.current_user.me().user_name
    parent = f"/Workspace/Users/{user}/genie_spaces"
    w.workspace.mkdirs(parent)
    created = w.api_client.do("POST", "/api/2.0/genie/spaces", body={**body, "parent_path": parent})
    SPACE_ID = created["space_id"]
    print("CREATED new space", SPACE_ID)

spaces with this exact title: ['01f1db9edffa1a298195db2806ce45e3']
UPDATED existing space 01f1db9edffa1a298195db2806ce45e3


## 3. Check what was saved
Read the space back and confirm the tables, instructions and examples, and that no held-out benchmark question is in it.

In [4]:
got = w.api_client.do("GET", f"/api/2.0/genie/spaces/{SPACE_ID}", query={"include_serialized_space": "true"})
saved = json.loads(got["serialized_space"])
print("title:", got["title"])
print("warehouse:", got["warehouse_id"])
print("description:", got.get("description"))
print("tables:", len(saved["data_sources"]["tables"]))
print("example SQLs:", len(saved["instructions"]["example_question_sqls"]))
print("sample questions:", len(saved["config"]["sample_questions"]))
ti = saved["instructions"]["text_instructions"]
print("instruction blocks:", len(ti), "| characters:", sum(len(x) for b in ti for x in b["content"]))
assert saved["data_sources"]["tables"] and len(saved["instructions"]["example_question_sqls"]) == 10

with open("benchmark_expected.json") as f:
    heldout = [q["question"] for q in json.load(f)["heldout"]]
blob = json.dumps(saved, ensure_ascii=False).lower()
leaked = [q for q in heldout if q.lower() in blob]
print("held-out questions found in the space:", leaked)
assert not leaked
print("OK: the 5 held-out questions are not in the space")

print()
print("Example questions loaded:")
for e in saved["instructions"]["example_question_sqls"]:
    print("  -", e["question"][0])

title: Blue Ridge LNG — PM Co-Pilot (synthetic)
warehouse: 527214d7319cb124
description: Ask about drawing compliance, schedule, risk, cost, safety and NCRs for the Blue Ridge LNG Terminal Phase 2 build. All data is synthetic (demo).
tables: 11
example SQLs: 10
sample questions: 10
instruction blocks: 1 | characters: 2625
held-out questions found in the space: []
OK: the 5 held-out questions are not in the space

Example questions loaded:
  - How many drawings failed compliance review this week, out of how many reviewed?
  - Show failed compliance findings this week by code citation.
  - Which piping isometric drawings failed compliance this week?
  - Which milestones are already late, and by how many days?
  - What are the top 3 highest-risk milestones in the next 90 days?
  - Explain the budget variance: how far over budget are we?
  - How many open NCRs do we have, by category?
  - What is our safety record: TRIR and days without an incident?
  - What does OSHA 1910.147 require?
  -

## 4. Write the space id

In [5]:
os.makedirs("outputs", exist_ok=True)
with open("outputs/space_id.txt", "w") as f:
    f.write(SPACE_ID + "\n")
print("SPACE_ID =", SPACE_ID)
print("URL: " + w.config.host.rstrip("/") + "/genie/rooms/" + SPACE_ID)

SPACE_ID = 01f1db9edffa1a298195db2806ce45e3
URL: https://fevm-mfg-mc-se-sa.cloud.databricks.com/genie/rooms/01f1db9edffa1a298195db2806ce45e3
